In [ ]:
# ─── 0. Imports ──────────────────────────────────────────────────────────────
import os, random, json, time
import numpy as np
import cv2
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
from facenet_pytorch import MTCNN          # pip install facenet-pytorch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# **Fixed random seed=42 everywhere**

In [ ]:
# ─── 1. Reproducibility ──────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")

In [ ]:
# ─── 2. Configuration ────────────────────────────────────────────────────────
CFG = dict(
    data_root    = "/content/data/FF++",      # adjust to your Colab path
    frames_dir   = "/content/frames_mtcnn",   # cached extracted frames
    real_dir     = "real",
    fake_dir     = "fake",
    frames_per_video = 10,
    img_size     = 224,
    batch_size   = 32,
    lr           = 1e-4,
    weight_decay = 1e-4,
    max_epochs   = 20,
    lr_patience  = 3,
    es_patience  = 5,
    dropout1     = 0.50,
    dropout2     = 0.25,
    train_ratio  = 0.70,
    val_ratio    = 0.15,
    test_ratio   = 0.15,
    output_dir   = "/content/outputs",
)
os.makedirs(CFG["frames_dir"], exist_ok=True)
os.makedirs(CFG["output_dir"], exist_ok=True)

# **MTCNN face detection instead of raw frames**

In [ ]:
# ─── 3. IMPROVEMENT #1 & #2: MTCNN Face Extraction ──────────────────────────
def extract_frames_mtcnn(video_path: str, out_dir: str,
                          n_frames: int = 10, img_size: int = 224) -> list[str]:
    """
    Extract n_frames evenly-spaced frames from a video,
    crop face with MTCNN, fall back to center-crop on failure.
    Returns list of saved JPEG paths.
    """
    mtcnn = MTCNN(image_size=img_size, margin=20, keep_all=False,
                  device=DEVICE, post_process=False)

    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        return []

    # IMPROVEMENT #3: linspace for uniform temporal coverage
    indices = np.linspace(0, total - 1, n_frames, dtype=int)

    stem   = Path(video_path).stem
    saved  = []
    for fi in indices:
        out_path = os.path.join(out_dir, f"{stem}_f{fi:04d}.jpg")
        if os.path.exists(out_path):          # IMPROVEMENT #8: cached
            saved.append(out_path)
            continue

        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read()
        if not ret:
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Try MTCNN face detection
        from PIL import Image as PILImage
        pil_img = PILImage.fromarray(frame_rgb)
        try:
            face = mtcnn(pil_img)              # returns tensor or None
            if face is not None:
                face_np = face.permute(1, 2, 0).numpy().astype(np.uint8)
                cv2.imwrite(out_path, cv2.cvtColor(face_np, cv2.COLOR_RGB2BGR))
                saved.append(out_path)
                continue
        except Exception:
            pass

        # IMPROVEMENT #2: Fallback – center crop
        h, w = frame_rgb.shape[:2]
        s    = min(h, w)
        top  = (h - s) // 2
        left = (w - s) // 2
        crop = frame_rgb[top:top+s, left:left+s]
        crop = cv2.resize(crop, (img_size, img_size))
        cv2.imwrite(out_path, cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
        saved.append(out_path)

    cap.release()
    return saved


def build_frame_dataset(cfg: dict) -> tuple[list, list]:
    """Iterate all videos, extract frames, return (paths, labels)."""
    all_paths, all_labels = [], []
    for label_name, label_int in [(cfg["real_dir"], 0), (cfg["fake_dir"], 1)]:
        vid_dir = os.path.join(cfg["data_root"], label_name)
        if not os.path.isdir(vid_dir):
            print(f"[WARN] Directory not found: {vid_dir}")
            continue
        for vid in sorted(os.listdir(vid_dir)):
            vpath = os.path.join(vid_dir, vid)
            frames = extract_frames_mtcnn(
                vpath, cfg["frames_dir"],
                n_frames=cfg["frames_per_video"],
                img_size=cfg["img_size"]
            )
            all_paths.extend(frames)
            all_labels.extend([label_int] * len(frames))

    print(f"[INFO] Total frames: {len(all_paths)}  "
          f"(real={all_labels.count(0)}, fake={all_labels.count(1)})")
    return all_paths, all_labels

In [ ]:
# ─── 4. Dataset & DataLoaders ────────────────────────────────────────────────
class DeepfakeDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        from PIL import Image as PILImage
        img = PILImage.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def make_loaders(paths, labels, cfg):
    # IMPROVEMENT #4: fixed seed for reproducible splits
    X_trv, X_te, y_trv, y_te = train_test_split(
        paths, labels, test_size=cfg["test_ratio"],
        stratify=labels, random_state=SEED)
    val_frac = cfg["val_ratio"] / (1 - cfg["test_ratio"])
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_trv, y_trv, test_size=val_frac,
        stratify=y_trv, random_state=SEED)

    tr_ds  = DeepfakeDataset(X_tr,  y_tr,  train_tf)
    val_ds = DeepfakeDataset(X_val, y_val, eval_tf)
    te_ds  = DeepfakeDataset(X_te,  y_te,  eval_tf)

    g = torch.Generator(); g.manual_seed(SEED)
    tr_dl  = DataLoader(tr_ds,  batch_size=cfg["batch_size"], shuffle=True,  num_workers=2, generator=g)
    val_dl = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False, num_workers=2)
    te_dl  = DataLoader(te_ds,  batch_size=cfg["batch_size"], shuffle=False, num_workers=2)

    print(f"[INFO] Train={len(tr_ds)} | Val={len(val_ds)} | Test={len(te_ds)}")
    return tr_dl, val_dl, te_dl

In [ ]:

# ─── 5. Model ────────────────────────────────────────────────────────────────
class DeepfakeCatcher(nn.Module):
    """
    Dual-Branch CNN Fusion (EfficientNet-B4 + XceptionNet).
    Feature-level concatenation → 3840-dim → MLP head → binary output.
    """
    def __init__(self, dropout1=0.5, dropout2=0.25):
        super().__init__()
        # Branch A: EfficientNet-B4  → 1792-dim
        self.eff = timm.create_model("efficientnet_b4", pretrained=True, num_classes=0)
        # Branch B: XceptionNet      → 2048-dim
        self.xcp = timm.create_model("xception", pretrained=True,
                                     num_classes=0, global_pool="avg")

        fused_dim = 1792 + 2048   # 3840

        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout1),
            nn.Linear(1024, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout2),
            nn.Linear(256, 2),
        )

    def forward(self, x):
        f_eff = self.eff(x)   # (B, 1792)
        f_xcp = self.xcp(x)   # (B, 2048)
        fused = torch.cat([f_eff, f_xcp], dim=1)   # (B, 3840)
        return self.classifier(fused)


In [ ]:
# ─── 6. Training Utilities ───────────────────────────────────────────────────
def run_epoch(model, loader, criterion, optimizer, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels, all_probs = 0., [], [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = logits.argmax(dim=1)

            total_loss  += loss.item() * imgs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

    n    = len(all_labels)
    acc  = accuracy_score(all_labels, all_preds) * 100
    auc  = roc_auc_score(all_labels, all_probs) * 100
    # IMPROVEMENT #5: weighted F1
    f1   = f1_score(all_labels, all_preds, average="weighted") * 100
    return total_loss / n, acc, auc, f1, all_labels, all_probs, all_preds


def plot_training_curves(history, out_dir):
    """Save Loss, Accuracy, and AUC-ROC training curves."""
    epochs = range(1, len(history["train_loss"]) + 1)
    best_e = history["best_epoch"]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Deepfake Catcher — Training Curves", fontsize=14, fontweight="bold")

    for ax, (tr_key, val_key, ylabel, title) in zip(axes, [
        ("train_loss", "val_loss", "Cross-Entropy Loss", "Loss"),
        ("train_acc",  "val_acc",  "Accuracy (%)",       "Accuracy"),
        ("val_auc",    "val_auc",  "AUC-ROC (%)",        "AUC-ROC"),
    ]):
        ax.plot(epochs, history[tr_key],  'b-', lw=2, label="Train")
        ax.plot(epochs, history[val_key], 'r-', lw=2, label="Val")
        if "auc" in tr_key:
            ax.axhline(history["test_auc"], color="orange", ls="--",
                       lw=1.5, label=f"Test AUC: {history['test_auc']:.2f}%")
        ax.axvline(best_e, color="green", ls="--", lw=1.5,
                   label=f"Best epoch ({best_e})")
        ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
        ax.set_title(title); ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "training_curves.png"), dpi=150, bbox_inches="tight")
    plt.close()


def plot_confusion_matrix(y_true, y_pred, out_dir):
    cm  = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    im  = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Real", "Fake"]); ax.set_yticklabels(["Real", "Fake"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix — Test Set")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black",
                    fontsize=14, fontweight="bold")
    plt.colorbar(im)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
    plt.close()


In [ ]:
# ─── 7. Main Training Loop ───────────────────────────────────────────────────
def train(cfg):
    # Build dataset
    paths, labels = build_frame_dataset(cfg)
    tr_dl, val_dl, te_dl = make_loaders(paths, labels, cfg)

    model     = DeepfakeCatcher(cfg["dropout1"], cfg["dropout2"]).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    # IMPROVEMENT #7: 'max' mode targeting val_accuracy
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=cfg["lr_patience"], verbose=True)

    history = {k: [] for k in
               ["train_loss","val_loss","train_acc","val_acc","val_auc","train_f1","val_f1"]}
    best_val_acc = 0.
    best_epoch   = 1
    patience_cnt = 0
    best_ckpt    = os.path.join(cfg["output_dir"], "best_deepfake_catcher.pth")

    print(f"\n{'Epoch':>5} {'TrLoss':>8} {'TrAcc':>8} {'VaLoss':>8} {'VaAcc':>8} "
          f"{'VaAUC':>8} {'VaF1':>8} {'LR':>10}")
    print("-" * 75)

    for epoch in range(1, cfg["max_epochs"] + 1):
        tr_loss, tr_acc, _, tr_f1, *_ = run_epoch(model, tr_dl, criterion, optimizer, train=True)
        va_loss, va_acc, va_auc, va_f1, *_ = run_epoch(model, val_dl, criterion, optimizer, train=False)

        scheduler.step(va_acc)       # IMPROVEMENT #7
        cur_lr = optimizer.param_groups[0]["lr"]

        # IMPROVEMENT #10: structured epoch log
        print(f"{epoch:>5} {tr_loss:>8.4f} {tr_acc:>7.2f}% {va_loss:>8.4f} "
              f"{va_acc:>7.2f}% {va_auc:>7.2f}% {va_f1:>7.2f}% {cur_lr:>10.2e}")

        for k, v in zip(["train_loss","val_loss","train_acc","val_acc","val_auc","train_f1","val_f1"],
                         [tr_loss, va_loss, tr_acc, va_acc, va_auc, tr_f1, va_f1]):
            history[k].append(v)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_epoch   = epoch
            patience_cnt = 0
            torch.save(model.state_dict(), best_ckpt)
        else:
            patience_cnt += 1
            if patience_cnt >= cfg["es_patience"]:
                print(f"\n[INFO] Early stopping at epoch {epoch}. Best epoch: {best_epoch}")
                break

    # ── Evaluate on test set ─────────────────────────────────────────────────
    model.load_state_dict(torch.load(best_ckpt))
    te_loss, te_acc, te_auc, te_f1, y_true, y_prob, y_pred = \
        run_epoch(model, te_dl, criterion, optimizer, train=False)

    history["test_auc"]   = te_auc
    history["best_epoch"] = best_epoch

    print("\n" + "=" * 60)
    print(f"  TEST RESULTS (best epoch {best_epoch})")
    print(f"  Accuracy  : {te_acc:.2f}%")
    print(f"  AUC-ROC   : {te_auc:.2f}%")
    print(f"  F1-Score  : {te_f1:.2f}%")
    print("=" * 60)

    # Save plots
    plot_training_curves(history, cfg["output_dir"])
    plot_confusion_matrix(y_true, y_pred, cfg["output_dir"])

    # Save metrics JSON
    metrics = {"accuracy": te_acc, "auc_roc": te_auc, "f1_score": te_f1,
               "best_epoch": best_epoch}
    with open(os.path.join(cfg["output_dir"], "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    print(f"[INFO] Outputs saved to: {cfg['output_dir']}")
    return metrics


if __name__ == "__main__":
    metrics = train(CFG)